# 🎯 Stage 2: Conditional Fine-TuningFine-tunes `CVanillaMolGen_RNN` on paired ligand-protein data from BindingDB.This bridges the gap between unconditional pretraining and conditional generation.**Prerequisites:** Stage 1 (pretraining) must be complete.

In [ ]:
import os, sysCTDDG_ROOT = os.environ.get("CTDDG_ROOT", os.path.dirname(os.getcwd()))os.environ["CTDDG_ROOT"] = CTDDG_ROOTos.environ["MXNET_CUDNN_LIB_CHECKING"] = "0"os.chdir(CTDDG_ROOT)import mxnet as mxNUM_GPUS = mx.context.num_gpus()print(f"Project root: {CTDDG_ROOT}")print(f"GPUs available: {NUM_GPUS}")!nvidia-smi --query-gpu=index,name --format=csv,noheader

## Verify Pretrained Checkpoint

In [ ]:
ckpt_path = os.path.join(CTDDG_ROOT, "outputs", "pretrain", "logs", "ckpt.params")config_path = os.path.join(CTDDG_ROOT, "outputs", "pretrain", "logs", "configs.json")for label, p in [("Checkpoint", ckpt_path), ("Config", config_path)]:    exists = os.path.exists(p)    print(f"  {'✅' if exists else '❌'} {label}: {p}")    if not exists:        raise FileNotFoundError(f"Missing: {p}. Run Stage 1 first!")

## Configure Fine-TuningSet which dataset to fine-tune on. The default is Dataset 1.

In [ ]:
# ── CONFIGURATION ──DATASET_INDEX = 1       # Which BindingDB dataset (1-5)ITERATIONS = 50000      # Fine-tuning iterationsBATCH_SIZE = 16         # Batch size per GPULEARNING_RATE = 1e-4SAVE_FREQ = 5000        # Checkpoint every N stepsLOG_FREQ = 100          # Log every N stepsprint(f"Dataset:    {DATASET_INDEX}")print(f"Iterations: {ITERATIONS:,}")print(f"Batch size: {BATCH_SIZE} × {NUM_GPUS} GPUs = {BATCH_SIZE * NUM_GPUS} effective")

## Run Fine-TuningThis executes the pretraining notebook first (to define all classes), then runs the fine-tuning code.

In [ ]:
%%time# The finetune_cell.py depends on classes defined in pretraining.ipynb.# We run pretraining.ipynb code (definitions only, not training) then the finetuning.# First, execute pretraining notebook to get all class definitions loadedprint("Loading model definitions from pretraining.ipynb...")exec_globals = {"__name__": "__main__"}# We need to run the pretraining notebook's code to define all the classes# but skip the actual training loop. We do this by loading it as a module.import jsonnb_path = os.path.join(CTDDG_ROOT, "code", "pretraining.ipynb")with open(nb_path) as f:    nb = json.load(f)# Get the source code from the single cellfull_src = "".join(nb["cells"][0]["source"])# Split at the training section and only execute definitions# Find where training startstrain_marker = '""\"# Training the model""\"'alt_marker = 'print("We are in training part....")'# Execute everything up to the training loopif alt_marker in full_src:    defs_src = full_src[:full_src.index(alt_marker)]elif "Training the model" in full_src:    idx = full_src.index("Training the model")    # Go back to find the triple-quote before it    defs_src = full_src[:idx-3]else:    print("⚠️ Could not find training marker, loading full source")    defs_src = full_src# Execute definitions in current namespaceexec(compile(defs_src, "<pretraining_defs>", "exec"))print("✅ Model definitions loaded")# Now run the fine-tuning codeprint("\nStarting fine-tuning...")exec(open(os.path.join(CTDDG_ROOT, "scripts", "finetune_cell.py")).read())run_finetuning(dataset_index=DATASET_INDEX)

## Check Fine-Tuning Output

In [ ]:
ft_dir = os.path.join(CTDDG_ROOT, "outputs", "CTDGD", f"Dataset{DATASET_INDEX}", "model")if os.path.exists(ft_dir):    for f in os.listdir(ft_dir):        fpath = os.path.join(ft_dir, f)        print(f"  ✅ {f} ({os.path.getsize(fpath)/1024:.1f} KB)")else:    print(f"❌ Fine-tuning output directory not found: {ft_dir}")